# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring a dataset using the `mlcroissant` library, referencing all entities by their `@id` according to the FAIR^2 Croissant schema.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the FAIR^2 dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access the metadata (ensure it's a single object)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs following the Croissant schema. All references are made by their `@id` as required.

In [ ]:
# List all available record sets by their @id
record_sets_info = dataset.get_record_sets()

print("Available Record Sets (@id):")
for rs in record_sets_info:
    print(f"@id: {rs['@id']} | name: {rs['name']}")

# Let's select the main survey record set for further analysis.
# For demonstration, we will assume the primary record set has an id:
# 'https://api.app.sen.science/frontiers/7160411/mental-health-survey-recordSet'
# You can verify using the actual dataset.
main_record_set_id = record_sets_info[0]['@id']  # First available record set @id

# List field info for the primary record set
fields = dataset.get_fields(record_set=main_record_set_id)
print(f"\nFields in record set {main_record_set_id}:")
for field in fields:
    print(f"@id: {field['@id']} | name: {field['name']} | dataType: {field.get('dataType', None)}")

# List sample records by their @id
print(f"\nSample records from {main_record_set_id}:")
for record in dataset.records(record_set=main_record_set_id):
    print(record)
    break

## 3. Data Extraction
Load data from specific record sets into pandas DataFrames for analysis. Record sets and fields are referenced using their `@id`s.

In [ ]:
# Extract all available record sets into DataFrames
record_sets_ids = [rs['@id'] for rs in dataset.get_record_sets()]
dataframes = {}

for record_set_id in record_sets_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df

# Explore columns in main record set
print(f"Columns in main record set ({main_record_set_id}):")
print(dataframes[main_record_set_id].columns.tolist())

# Show data sample
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

Examples include removing outliers, transforming distributions, or grouping data by key attributes in preparation for further analysis.

**All fields are referenced by their `@id`.**

In [ ]:
# Identify a numeric field for analysis by its @id
# Example: GAD-7 score field @id: 'https://api.app.sen.science/frontiers/7160411/gad7_score_field'
# Example: group by 'level_of_education' @id: 'https://api.app.sen.science/frontiers/7160411/level_of_education_field'

# Find numeric field
gad7_field_id = None
education_field_id = None
for field in dataset.get_fields(record_set=main_record_set_id):
    if 'gad' in field['name'].lower():
        gad7_field_id = field['@id']
    if 'education' in field['name'].lower():
        education_field_id = field['@id']

if gad7_field_id is None:
    print("GAD-7 numeric field not found. Using sample column name.")
    # Fallback: guess by column name
    gad7_field_id = 'gad7_score' if 'gad7_score' in dataframes[main_record_set_id].columns else dataframes[main_record_set_id].columns[0]
if education_field_id is None:
    print("Education field not found. Using sample column name.")
    education_field_id = 'level_of_education' if 'level_of_education' in dataframes[main_record_set_id].columns else dataframes[main_record_set_id].columns[0]

# Filter: GAD-7 scores > 10
threshold = 10
filtered_df = dataframes[main_record_set_id][dataframes[main_record_set_id][gad7_field_id] > threshold]
print(f"Filtered records with {gad7_field_id} > {threshold}:")
print(filtered_df.head())

filtered_df[f"{gad7_field_id}_normalized"] = (filtered_df[gad7_field_id] - filtered_df[gad7_field_id].mean()) / filtered_df[gad7_field_id].std()
print(f"Normalized {gad7_field_id} for filtered records:")
print(filtered_df[[gad7_field_id, f"{gad7_field_id}_normalized"]].head())

# Group by education level
if education_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(education_field_id)[gad7_field_id].mean().reset_index()
    print(f"Grouped average {gad7_field_id} by {education_field_id}:")
    print(grouped_df.head())

## 5. Visualization
Visualize the GAD-7 score distribution and relationships with education level, referencing each field with its `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 5))
sns.histplot(dataframes[main_record_set_id][gad7_field_id], bins=20, kde=True)
plt.title(f'GAD-7 Score Distribution ({gad7_field_id})')
plt.xlabel('GAD-7 Score')
plt.ylabel('Count')
plt.show()

if education_field_id in dataframes[main_record_set_id].columns:
    plt.figure(figsize=(10, 6))
    sns.boxplot(x=dataframes[main_record_set_id][education_field_id], y=dataframes[main_record_set_id][gad7_field_id])
    plt.title(f'GAD-7 Scores by Education Level ({education_field_id})')
    plt.xlabel('Level of Education')
    plt.ylabel('GAD-7 Score')
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
In this notebook, we've explored the FAIR^2 Mental Health Survey Dataset using the `mlcroissant` library.

* All record sets, fields, and columns were referenced by their Croissant `@id`.
* We examined key demographic and mental health indicators, filtered by GAD-7 scores, normalized numeric data, and visualized group differences.
* This approach demonstrates AI-ready, FAIR-compliant analysis workflows using standardized schema referencing for reproducible, transparent research.

Further analyses can include cross-record set joins, more detailed subgroup analysis, and integration of additional Croissant metadata for enhanced interpretability.